# neuralgeom — a guided tour of both lenses

Runs the two lenses end-to-end on synthetic data (no bundled datasets needed).

**Environment:** needs the subspace/topology extras. From the repo root:
```
pip install -e '.[geom,topology]'      # or activate your `nsg` env + `pip install torch`
```
See `ENVIRONMENT.md` and run `python examples/check_env.py` if unsure. For a
non-interactive pass/fail over every branch, run `python examples/verify_all.py`.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import neuralgeom as ng
print("neuralgeom", ng.__version__)

## 1. The shared contract — `data.Trajectory`

Every generator and loader emits one object; every analysis consumes it. Here we
generate a *moving ring* rate-RNN (its subspace should trace a loop) and round-trip
it through HDF5.

In [ ]:
from neuralgeom.synth.subspace_rnn import SubspaceRNNConfig, make_trajectory
from neuralgeom.data import load_trajectory

cfg  = SubspaceRNNConfig(connectivity="ring", ring_moving=True, N=50,
                         n_trials=6, duration=1.5, dt=2e-3, noise_std=0.05, seed=3)
traj = make_trajectory(cfg)
traj.save("_tour_ring.h5")
print(traj)
print("reloaded:", load_trajectory("_tour_ring.h5").meta["generator"])

## 2. Subspace lens — embed → Riemannian kinematics → tangent-PCA

Track the k=1 subspace the activity occupies as a point on `Gr(1,N) = ℝP^{N-1}`,
then measure how it moves. The `speed·dt == step_dist` identity is a self-check;
tangent-PCA gives the intrinsic dimensionality of the subspace motion.

In [ ]:
from neuralgeom.subspace import (EmbedConfig, embed_from_trajectory,
                                 compute_kinematics, tangent_pca)
from neuralgeom.subspace.embed import subspace_drift

emb = embed_from_trajectory(traj, trial=0, cfg=EmbedConfig(k=1, win=50, stride=10))
kin = compute_kinematics(emb["frames"], emb["win_times"])
tp  = tangent_pca(emb["frames"])
print(f"median sv_gap (reliability) = {np.median(emb['sv_gap']):.1f}")
print(f"geodesic efficiency         = {kin['efficiency']:.2f}")
print(f"tangent-PCA dim (90% var)   = {int(np.searchsorted(tp['cum_evr'],0.90)+1)}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(emb["win_times"], subspace_drift(emb["frames"]))
ax[0].set(title="subspace drift  d(frame_0, frame_t)", xlabel="time (s)",
          ylabel="geodesic distance")
im = ax[1].imshow(np.nan_to_num(kin["curvature"])[None], aspect="auto")  # placeholder-safe
D = None
from neuralgeom.topology.persistence import single_trial_distances
D = single_trial_distances(traj, 0, EmbedConfig(k=1, win=50, stride=10))
ax[1].clear(); im = ax[1].imshow(D, origin="lower", cmap="magma",
     extent=[emb['win_times'][0], emb['win_times'][-1]]*2)
ax[1].set(title="geodesic self-distance (recurrence)", xlabel="time (s)", ylabel="time (s)")
fig.colorbar(im, ax=ax[1]); fig.tight_layout()

## 3. Topology — persistent homology (𝔽₂)

A persistent H1 feature = a loop. The moving ring should show one at k=1, in the
single-trial trajectory and/or the across-trial pooled cloud.

In [ ]:
from neuralgeom.subspace import PoolConfig
from neuralgeom.topology.persistence import pooled_distances, ph, top_life
Dp = pooled_distances(traj, PoolConfig(k=1, n_pool=150, fields=False))
dg_s, dg_p = ph(D, maxdim=1), ph(Dp, maxdim=1)
print(f"H1 persistence  single={top_life(dg_s[1]):.2f}  pooled={top_life(dg_p[1]):.2f}")
try:
    from persim import plot_diagrams
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    plot_diagrams(dg_s, ax=ax[0]); ax[0].set_title("single-trial")
    plot_diagrams(dg_p, ax=ax[1]); ax[1].set_title("pooled cloud"); fig.tight_layout()
except Exception as e:
    print("persim plot skipped:", e)

## 4. Pullback lens — feed-forward geometry `g = JᵀJ`

The other half of the library: how a differentiable map distorts its domain
(local volume magnification `√det g`).

In [ ]:
import torch, torch.nn as nn
from neuralgeom.geometry import PullbackGeometry
net = nn.Sequential(nn.Linear(3, 64), nn.Tanh(), nn.Linear(64, 12))
geo = PullbackGeometry(net)
X = torch.randn(200, 3)
vol = geo.volume_element(X).detach().numpy()
plt.figure(figsize=(5,3)); plt.hist(vol, bins=30); plt.xlabel(r"$\sqrt{\det g}$")
plt.ylabel("count"); plt.title("local volume magnification"); plt.tight_layout()
print("mean √det g =", float(vol.mean()))

## Next

- `python examples/verify_all.py` — pass/fail over **every** branch (contract, synth,
  geometry, spd, grassmann, subspace, topology, dec, direct, dynamics, tasks).
- `scripts/subspace/demo_subspace_pipeline.py` — the full subspace pipeline across
  all connectivity families and regimes.
- `HANDOFF.md` / `docs/MERGE_NOTES.md` — API map and how the two repos were merged.